In [1]:
%pip install arxiv


Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


C:\Users\mahen\AppData\Local\Temp\ipykernel_19420\2616709328.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [3]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [4]:

wiki.name


'wikipedia'

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
embeddings=OllamaEmbeddings(model="nomic-embed-text")
vectordb=FAISS.from_documents(documents,embeddings)
retriever=vectordb.as_retriever()
retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F3F720ABA0>, search_kwargs={})

In [6]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith_search",
                      "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [7]:
retriever_tool.name


'langsmith_search'

In [8]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [9]:
tools=[wiki,arxiv,retriever_tool]

In [10]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2", temperature=0)

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI assistant. "
        "Use the available tools when you need information from the user's documents."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [12]:
#Agents


In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful research assistant. For every user question, you must call "
        "all available tools exactly once: Wikipedia, ArXiv, and the LangSmith documentation "
        "retriever. Use their results as evidence, then provide one clear, concise answer. "
        "If a tool returns no useful result, mention that briefly and continue."
    )
)

In [ ]:
question = "What is LangSmith?"

# Execute every tool once, then ask the model to synthesize the evidence.
tool_results = {}
for tool_name, tool in [
    ("Wikipedia", wiki),
    ("ArXiv", arxiv),
    ("LangSmith documentation", retriever_tool),
]:
    try:
        tool_results[tool_name] = tool.invoke(question)
    except Exception as error:
        tool_results[tool_name] = f"Tool failed: {type(error).__name__}: {error}"

evidence = "\n\n".join(
    f"{tool_name}:\n{result}"
    for tool_name, result in tool_results.items()
)

answer = llm.invoke(
    "Answer the user's question using the tool results below. "
    "Clearly distinguish documented facts from missing or inconclusive evidence.\n\n"
    f"User question: {question}\n\nTool results:\n{evidence}"
)

print("Tools called:", ", ".join(tool_results))
print("\nAnswer:\n")
print(answer.content)